# HW03 Problem 3 — FashionMNIST Image Classifier (PyTorch)

This notebook builds a multilayer perceptron (MLP) image classifier for the
FashionMNIST dataset using PyTorch, torchvision, and TorchMetrics, as requested:

- Load FashionMNIST and split the training data into a training set and a validation set.
- Build DataLoaders.
- Define a classifier (`nn.Module`): flattened 28x28 input → hidden layer (300, ReLU) → hidden layer (100, ReLU) → output layer (10 classes).
- Train with cross-entropy loss and an SGD optimizer for 20 epochs, tracking training loss, training accuracy, and validation accuracy per epoch.
- Plot training accuracy over the epochs.
- Evaluate the trained model on a few validation samples (predicted classes, class names, true labels, class probabilities, and top-k classes).

Requires Python 3.10 or higher.

Submitted by: Ponprom Rojanakiratikan

## 1. Setup: imports, Python version check, device, and random seed

Imports PyTorch, torchvision, TorchMetrics, and Matplotlib; confirms the Python
version is 3.10 or higher; selects a GPU if one is available (otherwise CPU);
and fixes the random seed so the split and training are reproducible.

In [ ]:
import sys

assert sys.version_info >= (3, 10), "This notebook requires Python 3.10 or higher."

import matplotlib.pyplot as plt
import torch
import torchmetrics
import torchvision
import torchvision.transforms.v2 as T
from torch import nn
from torch.utils.data import DataLoader, random_split

print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"TorchMetrics: {torchmetrics.__version__}")

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Device      : {device}")

SEED = 42
torch.manual_seed(SEED)

## 2. Load the FashionMNIST dataset

Downloads FashionMNIST (60,000 training images and 10,000 test images of
28x28 grayscale clothing items, 10 classes) via `torchvision.datasets`.
Each image is converted to a float tensor with pixel values scaled to [0, 1].

In [ ]:
toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=toTensor
)
test_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=toTensor
)

class_names = train_and_valid_data.classes
print(f"Training+validation images: {len(train_and_valid_data)}")
print(f"Test images               : {len(test_data)}")
print(f"Image shape               : {tuple(train_and_valid_data[0][0].shape)}")
print(f"Classes                   : {class_names}")

## 3. Split the training data into training and validation sets

Splits the 60,000 original training images into 55,000 for training and
5,000 for validation, using a seeded generator so the split is reproducible.

In [ ]:
train_data, valid_data = random_split(
    train_and_valid_data, [55_000, 5_000], generator=torch.Generator().manual_seed(SEED)
)
print(f"Training set  : {len(train_data)}")
print(f"Validation set: {len(valid_data)}")

## 4. Build the DataLoaders

Wraps each dataset in a `DataLoader` that serves mini-batches of 32 images.
The training loader shuffles every epoch; the validation and test loaders do not.

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

X_batch, y_batch = next(iter(train_loader))
print(f"One training batch: X {tuple(X_batch.shape)}, y {tuple(y_batch.shape)}")

## 5. Define the classifier (`nn.Module`)

Defines the MLP classifier:

| Layer | Details |
|---|---|
| Input | Flatten each 28x28 image into 784 features |
| Hidden 1 | Linear 784 → 300, ReLU |
| Hidden 2 | Linear 300 → 100, ReLU |
| Output | Linear 100 → 10 (one logit per class) |

The output layer returns raw logits; `nn.CrossEntropyLoss` applies softmax internally.

In [ ]:
class FashionMNISTClassifier(nn.Module):
    def __init__(self, n_inputs=28 * 28, n_hidden1=300, n_hidden2=100, n_classes=10):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes),
        )

    def forward(self, X):
        return self.mlp(X)


torch.manual_seed(SEED)
model = FashionMNISTClassifier().to(device)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 6. Define the loss function, optimizer, and accuracy metric

- Loss: cross-entropy (`nn.CrossEntropyLoss`).
- Optimizer: stochastic gradient descent (`torch.optim.SGD`) with learning rate 0.02.
- Metric: multiclass accuracy from TorchMetrics.

In [ ]:
LEARNING_RATE = 0.02
N_EPOCHS = 20

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

## 7. Training and evaluation functions

- `evaluate()` computes a metric (here, accuracy) over a whole DataLoader with the model in evaluation mode and gradients disabled.
- `train()` runs the training loop for the requested number of epochs. For each epoch it records the mean training loss, the training accuracy (accumulated over all training batches), and the validation accuracy, and prints them.

In [ ]:
def evaluate(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute().item()


def train(model, optimizer, loss_fn, metric, train_loader, valid_loader, n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        model.train()
        metric.reset()
        total_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()
            metric.update(y_pred, y_batch)

        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(evaluate(model, valid_loader, metric))
        print(
            f"Epoch {epoch + 1:2d}/{n_epochs}, "
            f"train loss: {history['train_losses'][-1]:.4f}, "
            f"train accuracy: {history['train_metrics'][-1]:.2%}, "
            f"valid accuracy: {history['valid_metrics'][-1]:.2%}"
        )
    return history

## 8. Train the model for 20 epochs

Runs the training loop for 20 epochs, printing the training loss, training
accuracy, and validation accuracy after every epoch.

In [ ]:
history = train(model, optimizer, loss_fn, accuracy, train_loader, valid_loader, N_EPOCHS)

## 9. Plot training accuracy over the epochs

Plots the training accuracy recorded at the end of each of the 20 epochs.

In [ ]:
epochs = range(1, N_EPOCHS + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_metrics"], marker="o", label="Training accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("FashionMNIST MLP: Training Accuracy per Epoch")
plt.xticks(list(epochs))
plt.grid(True)
plt.legend()
plt.show()

## 10. Evaluate the trained model on a few validation samples

Takes the first 5 samples from the validation set and, using the trained model:

- prints the predicted class index and class name for each sample,
- prints the true label index and class name,
- prints the full class-probability vector (softmax of the logits), and
- prints the top 3 most likely classes with their probabilities.

In [ ]:
N_SAMPLES = 5
TOP_K = 3

X_new, y_new = next(iter(valid_loader))
X_new, y_new = X_new[:N_SAMPLES].to(device), y_new[:N_SAMPLES].to(device)

model.eval()
with torch.no_grad():
    y_logits = model(X_new)
y_proba = torch.softmax(y_logits, dim=1)
y_pred = y_proba.argmax(dim=1)

print("Predicted classes  :", y_pred.tolist())
print("Predicted names    :", [class_names[i] for i in y_pred.tolist()])
print("True labels        :", y_new.tolist())
print("True names         :", [class_names[i] for i in y_new.tolist()])
print()
print("Class probabilities (rows = samples, columns = classes 0-9):")
print(y_proba.cpu().round(decimals=3))
print()

top_proba, top_classes = torch.topk(y_proba, k=TOP_K, dim=1)
for i in range(N_SAMPLES):
    top = ", ".join(
        f"{class_names[c]} ({p:.2%})"
        for c, p in zip(top_classes[i].tolist(), top_proba[i].tolist())
    )
    print(f"Sample {i}: true = {class_names[y_new[i].item()]:<12} | top-{TOP_K}: {top}")

### Visualize the evaluated validation samples

Shows the same validation images with their predicted and true class names
(green title = correct, red title = incorrect).

In [ ]:
fig, axes = plt.subplots(1, N_SAMPLES, figsize=(2.5 * N_SAMPLES, 3))
for i, ax in enumerate(axes):
    ax.imshow(X_new[i].squeeze().cpu(), cmap="binary")
    correct = y_pred[i].item() == y_new[i].item()
    ax.set_title(
        f"Pred: {class_names[y_pred[i].item()]}\nTrue: {class_names[y_new[i].item()]}",
        color="green" if correct else "red",
        fontsize=9,
    )
    ax.axis("off")
plt.tight_layout()
plt.show()